# 05 — Déployer et vérifier le prototype

Je déploie le **SFT mixte époque 3**, après rejet des variantes DPO. Ce carnet réunit le fonctionnement de l'application, les deux modes de service, la CI/CD et les preuves disponibles.

Les cellules relisent la configuration et des bilans ; elles ne démarrent aucun service ni GPU.
[04 · DPO](04_dpo.ipynb) · [Parcours](README.md) · [Rapport](../reports/rapport_technique_ocr14.md)


## 1. Du questionnaire à la proposition tracée

```text
cas fictif FR/EN → questionnaire → contrôle des champs
       → prompt partagé → Qwen + LoRA → contrôle ESI/priorité
       → proposition à valider OU revue humaine → trace de l'interaction
```

[domaine.py](../serving/domaine.py) définit les contrôles, [api.py](../serving/api.py) expose le service, [backend.py](../serving/backend.py) contacte le moteur d'inférence. Les journaux et la base SQLite permettent de retrouver une interaction ; ils ne constituent pas un dossier médical de production.

Un formulaire complet déclenche l'inférence. Une sortie invalide demande une revue humaine ; une sortie valide reste une proposition à vérifier. Les contrôles de format ne détectent pas toutes les hallucinations.


In [1]:
import json
from pathlib import Path

# Fonctionne depuis notebooks/ ou depuis la racine du dépôt.
RACINE = next(p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "serving/space/support/resultats.json").is_file())
def lire_json(relatif):
    return json.loads((RACINE / relatif).read_text(encoding="utf-8"))

resultats = lire_json("serving/space/support/resultats.json")
print("Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.")


Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.


In [2]:
modele = lire_json("serving/modele.json")
print("Modèle déclaré :", modele["id"])
print("Format SFT :", modele["format_sft"])
print("Révision et empreinte des poids conservées dans serving/modele.json.")


Modèle déclaré : ocr14-sft-mixte-ep3
Format SFT : template_sans_reflexion_eos_natif_v2
Révision et empreinte des poids conservées dans serving/modele.json.


## 2. Deux modes de service distincts

| | Interface publique | API optimisée |
|---|---|---|
| Hébergement | Hugging Face Space, ZeroGPU | VM GPU Colab temporaire ; Docker disponible |
| Stack | Gradio, Transformers, PEFT | FastAPI, vLLM, SQLite, tunnel HTTPS |
| Poids | SFT mixte époque 3 | Même adaptateur vérifié |
| Limite | Quotas et attente | Nouvelle URL à chaque session, arrêt automatique |

Le [Space](https://huggingface.co/spaces/rayakevin/ocr14-triage-fr-en) ne fonctionne pas sous vLLM. Le [lanceur Colab](06_lancer_demo_colab.ipynb) fournit la voie vLLM : paquet sur Drive, vérification des poids, démarrage du moteur et de l'API, tunnel, appels de contrôle, sauvegarde et arrêt. Il est séparé du parcours de lecture parce que son activation mobilise un GPU.

L'URL de la session du 10 novembre a expiré. Une démonstration en direct exige une nouvelle session et une nouvelle clé ; les preuves ci-dessous restent consultables sans ce lancement.


## 3. Ce qui a réellement été vérifié

| Contrôle | Résultat conservé | Ce qu'il ne prouve pas |
|---|---|---|
| Tests CPU | 52 tests lors de la vérification du 12 novembre | Justesse médicale du modèle |
| CI/CD | Tests, construction Docker, publication Space réussis | Disponibilité future du service |
| vLLM public, L4 le 10 novembre | Médiane HTTP 1,23 s sur trois cas fictifs | Performance clinique ou garantie de latence |
| Huit appels simultanés | Environ 1,54 s, soit 5,2 requêtes/s | Tenue en charge durable |
| Trace et sortie réelle | Une justification invente une hypotension malgré une TA de 150/90 | Absence d'hallucinations malgré un format valide |

Les anciens essais internes vLLM ont d'abord échoué sur un port occupé, puis réussi après correction. Ils sont résumés dans le [bilan historique](../reports/bilan_final_20261105.json) ; la preuve publique ultérieure concerne bien le modèle mixte.

Les traces publiques du 10 novembre sont archivées localement sous `data/evaluation/endpoint_public_20261110/`. Les sources des mesures et leurs limites sont détaillées dans le [rapport](../reports/rapport_technique_ocr14.md). Lire ce carnet ne réalise pas un nouveau test réseau.


## 4. Du dépôt au service : la CI/CD

```text
push GitHub → tests Pytest → construction Docker → publication contrôlée du Space
```

Le [workflow](../.github/workflows/poc.yml) conditionne le déploiement au succès des vérifications. [deployer.py](../serving/space/deployer.py) publie une liste fermée de fichiers et les poids vérifiés. Le jeton reste dans les secrets GitHub ; les corpus et notes privées ne sont pas publiés avec le Space.

Les [exécutions GitHub Actions](https://github.com/rayakevin/OCR_PROJET_14/actions/workflows/poc.yml) sont consultables. Les [instructions techniques](../serving/README.md) documentent les variables, Docker et les appels API. Elles constituent le mode d'emploi de reproduction ; ce carnet explique le fonctionnement et les résultats.


## 5. Conclusion et passage à l'échelle

La chaîne technique fonctionne : collecte bilingue, modèle adapté, contrôles, traçabilité et déploiement automatisé. Je ne revendique pas une validation clinique : sous-triage, références silver et hallucinations restent des limites majeures.

Pour aller plus loin : revue clinique indépendante et jeu de test neuf ; clarification des licences ; contrôle des biais ; authentification nominative et protection des traces ; tests de charge, disponibilité et coûts d'un hébergement durable.

**Parcours terminé : données → évaluation → SFT → DPO → déploiement.** Les expériences intermédiaires restent accessibles dans les archives, sans être nécessaires pour comprendre ce parcours.
